In [1]:
from survey_mars import SurveyMarsClient 
from survey_index import SurveyIndex 
from tips_parser import TipsParser 
client = SurveyMarsClient() 
client.authenticate() 
index = SurveyIndex(client).fetch() 
parser = TipsParser(client) # Save index 
index.save() # Loop through all surveys and save raw tips 
index.save_all_tips(parser)

Authenticating with Account ID and Secret...
Successfully authenticated! New tokens acquired.
Fetching survey list from SurveyMars...
Found 3 surveys (3 active, 0 closed)
Saved 3 surveys → data\raw\survey_index.json

Fetching responses for Melbourne GP (survey: 7UogehrrK)...
  Fetching page 1...
    Got 11 responses (total: 11)
Total responses fetched: 11
Saved 11 submissions → data\raw\tips\r01_melbourne_gp_tips.json

Player       Main race top 3                        Sprint                 DNFs
─────────────────────────────────────────────────────────────────────────────────────
Veljko       PIA → RUS → VER                        —                      LAW
Tara         PIA → VER → NOR                        —                      —
Riki         VER → RUS → NOR                        —                      —
Luca         RUS → VER → NOR                        —                      ALO, STR
Jake         RUS → LEC → VER                        —                      —
Alex         NOR 

[WindowsPath('data/raw/tips/r01_melbourne_gp_tips.json'),
 WindowsPath('data/raw/tips/r02_china_gp_tips.json'),
 WindowsPath('data/raw/tips/r03_japan_gp_tips.json')]

In [ ]:

index.fetch()
index.all()
index.save()

index = SurveyIndex(client).fetch()
parser = TipsParser(client)

Fetching survey list from SurveyMars...
Authenticating with Account ID and Secret...
Successfully authenticated! New tokens acquired.
Found 3 surveys (3 active, 0 closed)
Saved 3 surveys → data\raw\survey_index.json


WindowsPath('data/raw/survey_index.json')

In [3]:
index.all()

[{'round_num': 1,
  'survey_id': 'mYi12CABb',
  'title': 'Las Vegas GP (UAT)',
  'status': 2,
  'published_at': None,
  'response_count': 8},
 {'round_num': 2,
  'survey_id': '7UogeB9EK',
  'title': 'Las Vegas GP',
  'status': 2,
  'published_at': None,
  'response_count': 8},
 {'round_num': 3,
  'survey_id': 'iTazs9CfK',
  'title': 'Qatar GP',
  'status': 2,
  'published_at': None,
  'response_count': 8},
 {'round_num': 4,
  'survey_id': '3wCa5k0Ay',
  'title': 'Abu Dhabi GP',
  'status': 2,
  'published_at': None,
  'response_count': 8},
 {'round_num': 5,
  'survey_id': '7UogehrrK',
  'title': 'Melbourne GP',
  'status': 1,
  'published_at': '2026-03-03T21:48:25',
  'response_count': 11},
 {'round_num': 6,
  'survey_id': 'mnbiKPsAB',
  'title': 'China GP',
  'status': 1,
  'published_at': '2026-03-11T20:07:53',
  'response_count': 11},
 {'round_num': 7,
  'survey_id': 'm1BbiXKjA',
  'title': 'Japan GP',
  'status': 1,
  'published_at': '2026-03-24T21:17:30',
  'response_count': 11}]

In [23]:
"""
parse_tips.py
─────────────────────────────────────────────────────────────
Fetches responses for a given survey from SurveyMars, cleans
them into a consistent format, and saves to:

    data/raw/tips/r{round:02d}_{race_slug}_tips.json

Usage:
    python parse_tips.py --survey-id m1BbiXKjA \
                         --round 4 \
                         --race "Japan GP" \
                         --sprint        # include this flag on sprint weekends

Environment variables required:
    SURVEYMARS_ACCOUNT_ID
    SURVEYMARS_SECRET
"""

import os
import json
import argparse
from pathlib import Path
from datetime import datetime

# ── Assumes SurveyMarsClient is in the same directory ────────
from survey_mars import SurveyMarsClient


# ─────────────────────────────────────────────────────────────
# DRIVER CODE MAP
# 3-letter codes as returned by SurveyMars → full surname
# Add new drivers / variants as you discover them.
# ─────────────────────────────────────────────────────────────
DRIVER_MAP = {
    "NOR": "Norris",
    "PIA": "Piastri",
    "LEC": "Leclerc",
    "HAM": "Hamilton",
    "RUS": "Russell",
    "VER": "Verstappen",
    "ANT": "Antonelli",
    "HAD": "Hadjar",
    "SAI": "Sainz",
    "ALO": "Alonso",
    "ALB": "Albon",
    "GAS": "Gasly",
    "OCO": "Ocon",
    "STR": "Stroll",
    "HUL": "Hulkenberg",
    "TSU": "Tsunoda",
    "LAW": "Lawson",
    "BEA": "Bearman",
    "BOR": "Bortoleto",
    "LIN": "Lindblad",
    "DOO": "Doohan",
    "COL": "Colapinto",
    # Add more as needed
}

# ─────────────────────────────────────────────────────────────
# QUESTION INDEX CONSTANTS
# These match the question_index keys in the SurveyMars response.
# If you restructure your survey, update these.
# ─────────────────────────────────────────────────────────────
Q_NAME      = 10000          # "What is your name"
Q_RACE_POS  = range(20001, 20011)   # 20001–20010 = P1 through P10
Q_SPRINT_S1 = 30001          # Sprint P1 (update to match your survey)
Q_SPRINT_S2 = 30002          # Sprint P2
Q_SPRINT_S3 = 30003          # Sprint P3
Q_DNF       = 50000          # "Pick driver(s) to DNF"


def normalise_driver(code: str) -> str:
    """Convert a 3-letter code to a full surname. Falls back to the raw value."""
    if not code or code.strip() in ("", "(Empty)"):
        return None
    return DRIVER_MAP.get(code.strip().upper(), code.strip().title())


def normalise_player(name: str) -> str:
    return name.strip().title() if name and name.strip() else "Unknown"


def parse_dnf_answer(answer_text: str) -> list[str]:
    """
    DNF question returns either:
      - "(Empty)"  → no picks
      - A comma-separated string of codes e.g. "ALB,GAS"
      - A single code e.g. "ALB"
    Returns a list of normalised driver surnames.
    """
    if not answer_text or answer_text.strip() in ("", "(Empty)"):
        return []
    drivers = [normalise_driver(d.strip()) for d in answer_text.split(",")]
    return [d for d in drivers if d]  # filter out any None


def parse_response(raw_response: dict, is_sprint_weekend: bool) -> dict:
    """
    Convert one raw SurveyMars response dict into our clean storage format.

    Output shape:
    {
        "player":        "Luca",
        "response_id":   7,
        "submitted_at":  "2026-03-26T22:32:33",
        "time_spent_s":  154,
        "main_race":     ["Russell","Leclerc","Hamilton",...],  # P1–P10
        "sprint":        ["Norris","Piastri","Russell"] | null,
        "dnf_picks":     ["Albon"] | []
    }
    """
    items = raw_response.get("items", {})

    # ── Player name ───────────────────────────────────────────
    name_item = items.get(str(Q_NAME), {})
    player = normalise_player(name_item.get("answer_text", "Unknown"))

    # ── Main race picks: P1 → P10 ─────────────────────────────
    main_race = []
    for q_idx in Q_RACE_POS:
        item = items.get(str(q_idx), {})
        driver = normalise_driver(item.get("answer_text", ""))
        main_race.append(driver)  # None if blank/empty

    # ── Sprint picks (only on sprint weekends) ────────────────
    sprint = None
    if is_sprint_weekend:
        s_picks = []
        for q_idx in [Q_SPRINT_S1, Q_SPRINT_S2, Q_SPRINT_S3]:
            item = items.get(str(q_idx), {})
            s_picks.append(normalise_driver(item.get("answer_text", "")))
        # Only store sprint if at least one pick was made
        sprint = s_picks if any(s_picks) else None

    # ── DNF picks ─────────────────────────────────────────────
    dnf_item = items.get(str(Q_DNF), {})
    dnf_picks = parse_dnf_answer(dnf_item.get("answer_text", ""))

    return {
        "player":       player,
        "response_id":  raw_response.get("response_id"),
        "submitted_at": raw_response.get("time_submitted"),
        "time_spent_s": raw_response.get("time_spent"),
        "main_race":    main_race,
        "sprint":       sprint,
        "dnf_picks":    dnf_picks,
    }


def fetch_all_responses(client: SurveyMarsClient, survey_id: str) -> list[dict]:
    """Page through all responses for a survey and return flat list of response dicts."""
    all_responses = []
    page = 1
    page_size = 100

    while True:
        print(f"  Fetching page {page}...")
        data = client.make_request(
            "GET",
            f"surveys/{survey_id}/responses",
            params={"page_index": page, "page_size": page_size}
        )

        if not data.get("success"):
            raise RuntimeError(f"Failed to fetch responses: {data}")

        # Responses come back as a dict keyed by response_id
        answers_dict = data["data"]["answers"]

        # Each value in answers_dict is the response object
        batch = [v["value"] for v in answers_dict.values()]
        all_responses.extend(batch)
        print(f"    Got {len(batch)} responses (total so far: {len(all_responses)})")

        if not data["data"].get("has_next", False):
            break
        page += 1

    return all_responses


def main():
    parser = argparse.ArgumentParser(description="Fetch and parse SurveyMars tips")
    parser.add_argument("--survey-id", required=True,  help="SurveyMars survey ID e.g. m1BbiXKjA")
    parser.add_argument("--round",     required=True,  type=int, help="Race round number e.g. 4")
    parser.add_argument("--race",      required=True,  help="Race name e.g. 'Japan GP'")
    parser.add_argument("--sprint",    action="store_true", help="Flag if this is a sprint weekend")
    parser.add_argument("--force",     action="store_true", help="Re-fetch even if output already exists")
    args = parser.parse_args()

    race_slug = args.race.lower().replace(" ", "_")
    out_path  = Path(f"data/raw/tips/r{args.round:02d}_{race_slug}_tips.json")

    # Guard: don't overwrite existing raw data unless forced
    if out_path.exists() and not args.force:
        print(f"Tips already saved at {out_path}")
        print("Use --force to re-fetch and overwrite.")
        return

    # ── Authenticate ──────────────────────────────────────────
    client = SurveyMarsClient(
        account_id=os.environ["SURVEYMARS_ACCOUNT_ID"],
        secret=os.environ["SURVEYMARS_SECRET"],
    )
    client.authenticate()

    # ── Fetch ─────────────────────────────────────────────────
    print(f"\nFetching responses for survey {args.survey_id} ({args.race})...")
    raw_responses = fetch_all_responses(client, args.survey_id)
    print(f"Total responses fetched: {len(raw_responses)}")

    # ── Parse ─────────────────────────────────────────────────
    submissions = [parse_response(r, args.sprint) for r in raw_responses]

    # Warn about any unknown players
    unknown = [s for s in submissions if s["player"] == "Unknown"]
    if unknown:
        print(f"WARNING: {len(unknown)} submission(s) had no player name")

    # ── Save ──────────────────────────────────────────────────
    payload = {
        "round":            args.round,
        "race_name":        args.race,
        "survey_id":        args.survey_id,
        "is_sprint_weekend": args.sprint,
        "fetched_at":       datetime.utcnow().isoformat(),
        "total_responses":  len(submissions),
        "submissions":      submissions,
    }

    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(payload, indent=2))
    print(f"\nSaved {len(submissions)} submissions → {out_path}")

    # ── Preview ───────────────────────────────────────────────
    print("\nPreview of parsed submissions:")
    print(f"{'Player':<12} {'Main race top 3':<35} {'DNF picks'}")
    print("─" * 65)
    for s in submissions:
        top3   = " → ".join(str(d) for d in s["main_race"][:3])
        dnfs   = ", ".join(s["dnf_picks"]) if s["dnf_picks"] else "—"
        print(f"{s['player']:<12} {top3:<35} {dnfs}")

